In [37]:
import numpy as np

np.random.seed(42)
np.set_printoptions(precision=4, suppress=True)

def sigmoid(x): return 1 / (1 + np.exp(-np.clip(x, -50, 50)))

print("NumPy版本：", np.__version__)
print("环境准备完成")

NumPy版本： 2.0.2
环境准备完成


第二部分：手写简单 RNN

1，定义简单 RNN

In [38]:
class SimpleRNN:
    def __init__(self, input_size, hidden_size):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.Wx = np.random.randn(hidden_size, input_size) * 0.1
        self.Wh = np.random.randn(hidden_size, hidden_size) * 0.1
        self.b = np.zeros((hidden_size, 1))

    def forward(self, x, h_prev):
        input_part = self.Wx @ x
        memory_part = self.Wh @ h_prev
        h_next = np.tanh(input_part + memory_part + self.b)
        return h_next, input_part, memory_part

print("SimpleRNN类定义完成")

SimpleRNN类定义完成


2创建 RNN 并查看参数形状

In [39]:
input_size, hidden_size = 3, 4
rnn = SimpleRNN(input_size, hidden_size)

print("输入权重Wx形状：", rnn.Wx.shape)
print("隐藏状态权重Wh形状：", rnn.Wh.shape)
print("偏置b形状：", rnn.b.shape)
print("\n输入权重Wx：\n", rnn.Wx)
print("\n隐藏状态权重Wh：\n", rnn.Wh)

输入权重Wx形状： (4, 3)
隐藏状态权重Wh形状： (4, 4)
偏置b形状： (4, 1)

输入权重Wx：
 [[ 0.0497 -0.0138  0.0648]
 [ 0.1523 -0.0234 -0.0234]
 [ 0.1579  0.0767 -0.0469]
 [ 0.0543 -0.0463 -0.0466]]

隐藏状态权重Wh：
 [[ 0.0242 -0.1913 -0.1725 -0.0562]
 [-0.1013  0.0314 -0.0908 -0.1412]
 [ 0.1466 -0.0226  0.0068 -0.1425]
 [-0.0544  0.0111 -0.1151  0.0376]]


3运行一个时间步

In [40]:
x = np.array([[1.0], [0.5], [-0.2]])
h_prev = np.zeros((hidden_size, 1))

h_next, input_part, memory_part = rnn.forward(x, h_prev)

print("当前输入x：\n", x)
print("\n上一时刻隐藏状态h_prev：\n", h_prev)
print("\n当前输入产生的结果Wx@x：\n", input_part)
print("\n历史记忆产生的结果Wh@h_prev：\n", memory_part)
print("\n新的隐藏状态h_next：\n", h_next)

当前输入x：
 [[ 1. ]
 [ 0.5]
 [-0.2]]

上一时刻隐藏状态h_prev：
 [[0.]
 [0.]
 [0.]
 [0.]]

当前输入产生的结果Wx@x：
 [[0.0298]
 [0.1453]
 [0.2057]
 [0.0404]]

历史记忆产生的结果Wh@h_prev：
 [[0.]
 [0.]
 [0.]
 [0.]]

新的隐藏状态h_next：
 [[0.0298]
 [0.1443]
 [0.2028]
 [0.0404]]


4让 RNN 连续读取三个时间步

In [41]:
sequence = [np.array([[1.0], [0.5], [-0.2]]), np.array([[0.2], [1.0], [0.3]]), np.array([[-0.5], [0.4], [1.0]])]
h = np.zeros((hidden_size, 1))

for time_step, x_t in enumerate(sequence, start=1):
    h, input_part, memory_part = rnn.forward(x_t, h)
    print(f"第{time_step}个时间步")
    print("输入：", x_t.ravel())
    print("历史信息贡献：", memory_part.ravel())
    print("新的隐藏状态：", h.ravel())
    print("-" * 60)

第1个时间步
输入： [ 1.   0.5 -0.2]
历史信息贡献： [0. 0. 0. 0.]
新的隐藏状态： [0.0298 0.1443 0.2028 0.0404]
------------------------------------------------------------
第2个时间步
输入： [0.2 1.  0.3]
历史信息贡献： [-0.0641 -0.0226 -0.0033 -0.0219]
新的隐藏状态： [-0.0486 -0.0226  0.0907 -0.0712]
------------------------------------------------------------
第3个时间步
输入： [-0.5  0.4  1. ]
历史信息贡献： [-0.0085  0.006   0.0041 -0.0107]
新的隐藏状态： [ 0.0259 -0.1025 -0.0908 -0.1026]
------------------------------------------------------------


第三部分：手写 LSTM

1定义 LSTM 单元

In [42]:
class LSTMCell:
    def __init__(self, input_size, hidden_size):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.W = np.random.randn(4 * hidden_size, input_size + hidden_size) * 0.1
        self.b = np.zeros((4 * hidden_size, 1))

    def forward(self, x, h_prev, c_prev):
        combined = np.vstack((h_prev, x))
        gates = self.W @ combined + self.b
        f_raw, i_raw, o_raw, g_raw = np.split(gates, 4, axis=0)
        f_gate = sigmoid(f_raw)
        i_gate = sigmoid(i_raw)
        o_gate = sigmoid(o_raw)
        c_candidate = np.tanh(g_raw)
        c_next = f_gate * c_prev + i_gate * c_candidate
        h_next = o_gate * np.tanh(c_next)
        details = {"遗忘门": f_gate, "输入门": i_gate, "输出门": o_gate, "候选记忆": c_candidate}
        return h_next, c_next, details

print("LSTMCell类定义完成")

LSTMCell类定义完成


2创建 LSTM 并运行一个时间步

In [43]:
lstm = LSTMCell(input_size=3, hidden_size=4)
x = np.array([[1.0], [0.5], [-0.2]])
h_prev = np.zeros((4, 1))
c_prev = np.zeros((4, 1))

h_next, c_next, details = lstm.forward(x, h_prev, c_prev)

print("输入x：\n", x)
print("\n上一时刻隐藏状态h_prev：\n", h_prev)
print("\n上一时刻记忆状态c_prev：\n", c_prev)

for name, value in details.items():
    print(f"\n{name}：\n{value}")

print("\n新的记忆状态c_next：\n", c_next)
print("\n新的隐藏状态h_next：\n", h_next)

输入x：
 [[ 1. ]
 [ 0.5]
 [-0.2]]

上一时刻隐藏状态h_prev：
 [[0.]
 [0.]
 [0.]
 [0.]]

上一时刻记忆状态c_prev：
 [[0.]
 [0.]
 [0.]
 [0.]]

遗忘门：
[[0.4823]
 [0.5133]
 [0.5   ]
 [0.5235]]

输入门：
[[0.4912]
 [0.5328]
 [0.4446]
 [0.53  ]]

输出门：
[[0.4927]
 [0.4659]
 [0.4822]
 [0.4743]]

候选记忆：
[[-0.0846]
 [-0.1534]
 [-0.1297]
 [ 0.0331]]

新的记忆状态c_next：
 [[-0.0415]
 [-0.0817]
 [-0.0576]
 [ 0.0175]]

新的隐藏状态h_next：
 [[-0.0205]
 [-0.038 ]
 [-0.0278]
 [ 0.0083]]


3观察 LSTM 连续处理序列

In [44]:
sequence = [np.array([[1.0], [0.5], [-0.2]]), np.array([[0.2], [1.0], [0.3]]), np.array([[-0.5], [0.4], [1.0]])]
h = np.zeros((4, 1))
c = np.zeros((4, 1))

for time_step, x_t in enumerate(sequence, start=1):
    h, c, details = lstm.forward(x_t, h, c)
    print(f"第{time_step}个时间步")
    print("输入：", x_t.ravel())
    print("遗忘门：", details["遗忘门"].ravel())
    print("输入门：", details["输入门"].ravel())
    print("输出门：", details["输出门"].ravel())
    print("记忆状态：", c.ravel())
    print("隐藏状态：", h.ravel())
    print("-" * 60)

第1个时间步
输入： [ 1.   0.5 -0.2]
遗忘门： [0.4823 0.5133 0.5    0.5235]
输入门： [0.4912 0.5328 0.4446 0.53  ]
输出门： [0.4927 0.4659 0.4822 0.4743]
记忆状态： [-0.0415 -0.0817 -0.0576  0.0175]
隐藏状态： [-0.0205 -0.038  -0.0278  0.0083]
------------------------------------------------------------
第2个时间步
输入： [0.2 1.  0.3]
遗忘门： [0.4811 0.5222 0.5279 0.5365]
输入门： [0.4854 0.5081 0.5068 0.5362]
输出门： [0.5112 0.5024 0.4781 0.4496]
记忆状态： [-0.0645  0.0057 -0.057   0.0285]
隐藏状态： [-0.0329  0.0029 -0.0272  0.0128]
------------------------------------------------------------
第3个时间步
输入： [-0.5  0.4  1. ]
遗忘门： [0.5116 0.5111 0.5257 0.5274]
输入门： [0.4772 0.4751 0.5434 0.4989]
输出门： [0.5143 0.527  0.493  0.4809]
记忆状态： [ 0.0007  0.1476 -0.0429 -0.0256]
隐藏状态： [ 0.0003  0.0772 -0.0211 -0.0123]
------------------------------------------------------------


第四部分：手写 GRU

1定义 GRU 单元

In [45]:
class GRUCell:
    def __init__(self, input_size, hidden_size):
        self.input_size = input_size
        self.hidden_size = hidden_size
        combined_size = input_size + hidden_size
        self.Wz = np.random.randn(hidden_size, combined_size) * 0.1
        self.Wr = np.random.randn(hidden_size, combined_size) * 0.1
        self.Wh = np.random.randn(hidden_size, combined_size) * 0.1
        self.bz = np.zeros((hidden_size, 1))
        self.br = np.zeros((hidden_size, 1))
        self.bh = np.zeros((hidden_size, 1))

    def forward(self, x, h_prev):
        combined = np.vstack((h_prev, x))
        z = sigmoid(self.Wz @ combined + self.bz)
        r = sigmoid(self.Wr @ combined + self.br)
        candidate_input = np.vstack((r * h_prev, x))
        h_candidate = np.tanh(self.Wh @ candidate_input + self.bh)
        h_next = (1 - z) * h_prev + z * h_candidate
        details = {"更新门": z, "重置门": r, "候选隐藏状态": h_candidate}
        return h_next, details

print("GRUCell类定义完成")

GRUCell类定义完成


2运行一个 GRU 时间步

In [46]:
gru = GRUCell(input_size=3, hidden_size=4)
x = np.array([[1.0], [0.5], [-0.2]])
h_prev = np.zeros((4, 1))

h_next, details = gru.forward(x, h_prev)

print("输入x：\n", x)
print("\n上一时刻隐藏状态h_prev：\n", h_prev)

for name, value in details.items():
    print(f"\n{name}：\n{value}")

print("\n新的隐藏状态h_next：\n", h_next)

输入x：
 [[ 1. ]
 [ 0.5]
 [-0.2]]

上一时刻隐藏状态h_prev：
 [[0.]
 [0.]
 [0.]
 [0.]]

更新门：
[[0.5224]
 [0.499 ]
 [0.4833]
 [0.5111]]

重置门：
[[0.501 ]
 [0.5794]
 [0.5151]
 [0.4691]]

候选隐藏状态：
[[ 0.0421]
 [ 0.0001]
 [ 0.0219]
 [-0.1196]]

新的隐藏状态h_next：
 [[ 0.022 ]
 [ 0.    ]
 [ 0.0106]
 [-0.0611]]


3观察 GRU 连续处理序列

In [47]:
sequence = [np.array([[1.0], [0.5], [-0.2]]), np.array([[0.2], [1.0], [0.3]]), np.array([[-0.5], [0.4], [1.0]])]
h = np.zeros((4, 1))

for time_step, x_t in enumerate(sequence, start=1):
    h, details = gru.forward(x_t, h)
    print(f"第{time_step}个时间步")
    print("输入：", x_t.ravel())
    print("更新门：", details["更新门"].ravel())
    print("重置门：", details["重置门"].ravel())
    print("候选隐藏状态：", details["候选隐藏状态"].ravel())
    print("最终隐藏状态：", h.ravel())
    print("-" * 60)

第1个时间步
输入： [ 1.   0.5 -0.2]
更新门： [0.5224 0.499  0.4833 0.5111]
重置门： [0.501  0.5794 0.5151 0.4691]
候选隐藏状态： [ 0.0421  0.0001  0.0219 -0.1196]
最终隐藏状态： [ 0.022   0.      0.0106 -0.0611]
------------------------------------------------------------
第2个时间步
输入： [0.2 1.  0.3]
更新门： [0.511  0.4854 0.5031 0.5355]
重置门： [0.5111 0.524  0.4925 0.5001]
候选隐藏状态： [ 0.0979  0.1747  0.0455 -0.0244]
最终隐藏状态： [ 0.0608  0.0848  0.0282 -0.0415]
------------------------------------------------------------
第3个时间步
输入： [-0.5  0.4  1. ]
更新门： [0.4755 0.4937 0.4968 0.5524]
重置门： [0.5095 0.4535 0.4707 0.524 ]
候选隐藏状态： [ 0.1128  0.359  -0.0252 -0.0479]
最终隐藏状态： [ 0.0855  0.2202  0.0017 -0.045 ]
------------------------------------------------------------


第五部分：同时比较 RNN、LSTM、GRU

让三个模型处理相同序列

In [48]:
test_sequence = [np.array([[1.0], [0.5], [-0.2]]), np.array([[0.2], [1.0], [0.3]]), np.array([[-0.5], [0.4], [1.0]])]

rnn_h = np.zeros((4, 1))
lstm_h = np.zeros((4, 1))
lstm_c = np.zeros((4, 1))
gru_h = np.zeros((4, 1))

for time_step, x_t in enumerate(test_sequence, start=1):
    rnn_h, _, _ = rnn.forward(x_t, rnn_h)
    lstm_h, lstm_c, _ = lstm.forward(x_t, lstm_h, lstm_c)
    gru_h, _ = gru.forward(x_t, gru_h)
    print(f"第{time_step}个时间步")
    print("RNN隐藏状态： ", rnn_h.ravel())
    print("LSTM隐藏状态：", lstm_h.ravel())
    print("LSTM记忆状态：", lstm_c.ravel())
    print("GRU隐藏状态： ", gru_h.ravel())
    print("-" * 70)

第1个时间步
RNN隐藏状态：  [0.0298 0.1443 0.2028 0.0404]
LSTM隐藏状态： [-0.0205 -0.038  -0.0278  0.0083]
LSTM记忆状态： [-0.0415 -0.0817 -0.0576  0.0175]
GRU隐藏状态：  [ 0.022   0.      0.0106 -0.0611]
----------------------------------------------------------------------
第2个时间步
RNN隐藏状态：  [-0.0486 -0.0226  0.0907 -0.0712]
LSTM隐藏状态： [-0.0329  0.0029 -0.0272  0.0128]
LSTM记忆状态： [-0.0645  0.0057 -0.057   0.0285]
GRU隐藏状态：  [ 0.0608  0.0848  0.0282 -0.0415]
----------------------------------------------------------------------
第3个时间步
RNN隐藏状态：  [ 0.0259 -0.1025 -0.0908 -0.1026]
LSTM隐藏状态： [ 0.0003  0.0772 -0.0211 -0.0123]
LSTM记忆状态： [ 0.0007  0.1476 -0.0429 -0.0256]
GRU隐藏状态：  [ 0.0855  0.2202  0.0017 -0.045 ]
----------------------------------------------------------------------


第六部分：页面中的双向 LSTM

In [49]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Bidirectional, LSTM, Dense

tf.random.set_seed(42)

bilstm_model = Sequential([Input(shape=(5, 3)), Bidirectional(LSTM(4)), Dense(2, activation="softmax")])
bilstm_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional_2 (Bidirectional) │ (None, 8)              │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 2)              │            18 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 274 (1.07 KB)

 Trainable params: 274 (1.07 KB)

 Non-trainable params: 0 (0.00 B)

准备测试数据

In [50]:
x_batch = np.array([[[1.0, 0.5, -0.2], [0.2, 1.0, 0.3], [-0.5, 0.4, 1.0], [0.6, -0.3, 0.8], [0.1, 0.7, -0.4]], [[0.3, 0.2, 0.1], [0.5, -0.6, 0.4], [0.9, 0.1, -0.2], [-0.4, 0.8, 0.6], [0.2, -0.1, 0.7]]], dtype=np.float32)

print("输入数据形状：", x_batch.shape)
print("样本数量：", x_batch.shape[0])
print("时间步数量：", x_batch.shape[1])
print("每个时间步的特征数：", x_batch.shape[2])

输入数据形状： (2, 5, 3)
样本数量： 2
时间步数量： 5
每个时间步的特征数： 3


运行双向 LSTM

In [51]:
predictions = bilstm_model(x_batch, training=False).numpy()

print("模型输出形状：", predictions.shape)
print("分类概率：\n", predictions)
print("每个样本预测类别：", np.argmax(predictions, axis=1))

模型输出形状： (2, 2)
分类概率：
 [[0.5132 0.4868]
 [0.5011 0.4989]]
每个样本预测类别： [0 0]


第七部分：查看双向 LSTM 的中间结果

单独提取双向 LSTM 输出

In [53]:
# 直接调用双向LSTM层，提取中间特征
bilstm_features=bilstm_model.layers[0](x_batch,training=False).numpy()

print("第一层名称：",bilstm_model.layers[0].name)
print("双向LSTM特征形状：",bilstm_features.shape)
print("第一个样本的双向特征：\n",bilstm_features[0])

第一层名称： bidirectional_2
双向LSTM特征形状： (2, 8)
第一个样本的双向特征：
 [ 0.026   0.2264  0.1314 -0.0758 -0.02   -0.0318 -0.0279  0.0965]
